# N120 · 回文DP、中心扩展与Manacher

**目标：** 从对称复用推导线性回文半径算法。

**先修：** N098, N118。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 奇偶中心；回文区间DP；镜像中心；右边界；分隔符风险。

## 从问题到算法

回文可用三种状态理解：区间DP依赖去掉两端后的内部区间；中心扩展逐个中心比较对称字符；Manacher复用当前最右回文内部的镜像半径，只在越过已知右边界时继续比较。奇偶中心分开表示，比插入特殊字符更容易明确边界。

## 最小实现

**本章接口：** `longest_palindrome_center(s)`、`palindrome_dp(s)`、`manacher_radii(s)`、`count_palindromic_substrings(s)`

In [1]:
def longest_palindrome_center(s):
    best=(0,0)
    for center in range(len(s)):
        for l,r in [(center,center),(center,center+1)]:
            while l>=0 and r<len(s) and s[l]==s[r]: l-=1; r+=1
            if r-l-1>best[1]-best[0]: best=(l+1,r)
    return s[best[0]:best[1]]

def palindrome_dp(s):
    n=len(s); dp=[[False]*n for _ in range(n)]
    for l in range(n-1,-1,-1):
        for r in range(l,n): dp[l][r]=s[l]==s[r] and (r-l<2 or dp[l+1][r-1])
    return dp

def manacher_radii(s):
    n=len(s); odd=[0]*n; left=0; right=-1
    for i in range(n):
        radius=1 if i>right else min(odd[left+right-i],right-i+1)
        while i-radius>=0 and i+radius<n and s[i-radius]==s[i+radius]: radius+=1
        odd[i]=radius
        if i+radius-1>right: left,right=i-radius+1,i+radius-1
    even=[0]*n; left=0; right=-1
    for i in range(n):
        radius=0 if i>right else min(even[left+right-i+1],right-i+1)
        while i-radius-1>=0 and i+radius<n and s[i-radius-1]==s[i+radius]: radius+=1
        even[i]=radius
        if i+radius-1>right: left,right=i-radius,i+radius-1
    return odd,even

def count_palindromic_substrings(s):
    odd,even=manacher_radii(s); return sum(odd)+sum(even)

## 正确性、前提与复杂度

镜像回文在当前大回文右界以内的部分由对称性保证，超过右界的字符尚未知，必须显式比较。每次真正扩展使全局最右边界前进，累计O(n)。每个奇中心半径r贡献r个非空回文，每个偶中心同理，所以求和得到按位置计数而非去重字符串数。

**代价：** 中心扩展O(n²)时间O(1)辅助；区间DP O(n²)时间空间；Manacher O(n)时间空间。odd[i]包含中心字符，even[i]以i−1与i之间为中心。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='abacabba'; odd,even=manacher_radii(s)
show_table(['中心下标','奇半径(含中心)','偶半径(中心在i前)','最长奇回文','最长偶回文'],[(i,odd[i],even[i],s[i-odd[i]+1:i+odd[i]],s[i-even[i]:i+even[i]]) for i in range(len(s))])

中心下标,奇半径(含中心),偶半径(中心在i前),最长奇回文,最长偶回文
0,1,0,a,
1,2,0,aba,
2,1,0,a,
3,3,0,bacab,
4,1,0,a,
5,1,0,b,
6,1,2,b,abba
7,1,0,a,


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert longest_palindrome_center('babad') in ('bab','aba')
assert longest_palindrome_center('cbbd')=='bb' and count_palindromic_substrings('aaa')==6
assert longest_palindrome_center('')=='' and manacher_radii('')==([],[])
from itertools import product
for n in range(8):
    for chars in product('ab',repeat=n):
        s=''.join(chars); all_pal=[s[l:r] for l in range(n) for r in range(l+1,n+1) if s[l:r]==s[l:r][::-1]]
        odd,even=manacher_radii(s); table=palindrome_dp(s)
        assert count_palindromic_substrings(s)==len(all_pal)==sum(sum(row) for row in table)
        longest=max([0]+[len(p) for p in all_pal])
        assert len(longest_palindrome_center(s))==longest==max([0]+[2*x-1 for x in odd]+[2*x for x in even])
        for l in range(n):
            for r in range(l,n): assert table[l][r]==(s[l:r+1]==s[l:r+1][::-1])
print('N120: 所有本章断言通过')

N120: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 从半径统计全部回文数。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明分隔符可能出现在输入时如何处理。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 5. Longest Palindromic Substring（canonical）
- 647. Palindromic Substrings（canonical）
- 214. Shortest Palindrome（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。